In [27]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [28]:
os.makedirs("../processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

In [29]:
df = pd.read_csv("../dataset/smart_grid_dataset_v8_final.csv")

print(df.shape)

df.head()

(14852, 17)


,datetime,division,district,upazila,temperature,humidity,rainfall,risk_level,hour,day_of_week,month,wind_speed,electricity_demand_mw,generation_capacity_mw,outage_count_last_24h,transformer_load_percent,grid_stability_score
0,2026-04-13 00:00:00,Sylhet,Habiganj,Bahubal,21.706209,89.780714,0.162541,Low,-0.0,-0.0,4.0,2.369280,148.946760,138.207721,0.0,88.850669,81.895900
1,2026-04-13 00:00:00,Sylhet,Habiganj,Bahubal,22.123703,89.129705,0.025049,Medium,0.0,-0.0,4.0,4.164438,162.994387,126.025554,0.0,101.536708,74.544691
2,2026-04-13 00:00:00,Sylhet,Habiganj,Baniachong,22.739698,81.374526,0.000000,Medium,-0.0,-0.0,4.0,7.900071,190.250642,187.402652,0.0,99.727282,82.618463
3,2026-04-13 00:00:00,Sylhet,Habiganj,Chunarughat,21.660948,88.134157,0.123960,Low,-1.0,-0.0,4.0,5.276376,121.885863,124.506115,1.0,78.926983,79.933550
4,2026-04-13 00:00:00,Sylhet,Habiganj,Habiganj Sadar,22.036211,92.543279,0.033939,Medium,-0.0,0.0,4.0,4.231840,173.981050,165.016931,1.0,99.658110,72.631408


In [30]:
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 14852 entries, 0 to 14851
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   datetime                  14852 non-null  str    
 1   division                  14852 non-null  str    
 2   district                  14852 non-null  str    
 3   upazila                   14852 non-null  str    
 4   temperature               14852 non-null  float64
 5   humidity                  14852 non-null  float64
 6   rainfall                  14852 non-null  float64
 7   risk_level                14852 non-null  str    
 8   hour                      14852 non-null  float64
 9   day_of_week               14852 non-null  float64
 10  month                     14852 non-null  float64
 11  wind_speed                14852 non-null  float64
 12  electricity_demand_mw     14852 non-null  float64
 13  generation_capacity_mw    14852 non-null  float64
 14  outage_count_last

In [31]:
df = df.drop_duplicates()

print(df.shape)

(14852, 17)


In [32]:
df["datetime"] = pd.to_datetime(df["datetime"])

df["hour"] = df["datetime"].dt.hour
df["day"] = df["datetime"].dt.day
df["month"] = df["datetime"].dt.month
df["day_of_week"] = df["datetime"].dt.dayofweek

df.drop("datetime", axis=1, inplace=True)

In [33]:
cat_cols = [
    "division",
    "district",
    "upazila"
]

feature_encoders = {}

for col in cat_cols:

    le = LabelEncoder()

    df[col] = le.fit_transform(df[col])

    feature_encoders[col] = le

    # Save encoder
    joblib.dump(
        le,
        f"../processed/{col}_encoder.pkl"
    )

print("Feature encoders saved.")

Feature encoders saved.


In [34]:
target_encoder = LabelEncoder()

df["risk_level"] = target_encoder.fit_transform(
    df["risk_level"]
)

joblib.dump(
    target_encoder,
    "../processed/label_encoder.pkl"
)

print(target_encoder.classes_)

['High' 'Low' 'Medium']


In [35]:
df["demand_capacity_ratio"] = (
    df["electricity_demand_mw"] /
    (df["generation_capacity_mw"] + 1)
)

df["load_stability_ratio"] = (
    df["transformer_load_percent"] /
    (df["grid_stability_score"] + 1)
)

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

In [36]:
X = df.drop("risk_level", axis=1)

y = df["risk_level"]

print(X.shape)
print(y.shape)

(14852, 20)
(14852,)


In [37]:
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

joblib.dump(
    scaler,
    "../processed/scaler.pkl"
)

print("Scaler Saved")

Scaler Saved


In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(11881, 20)
(2971, 20)


In [39]:
np.save("../processed/X_train.npy", X_train)
np.save("../processed/X_test.npy", X_test)

np.save("../processed/y_train.npy", y_train)
np.save("../processed/y_test.npy", y_test)

joblib.dump(scaler, "../processed/scaler.pkl")
joblib.dump(target_encoder, "../processed/label_encoder.pkl")

print("All processed files saved.")

All processed files saved.
